# 01 · Data collection

This notebook builds the raw data for the event study. It saves everything to `data/` so later notebooks never need to hit the internet again.

**What we collect**

1. **Treatment events:** the day each Madden cover athlete was announced.
2. **Comparison events:** the day each AP First-Team All-Pro list was announced, plus every player on it.
3. **Outcome:** daily English Wikipedia page views for every player, from 90 days before to 180 days after their event.

**Why Wikipedia page views?** They're free and official, they come in daily counts going back to July 2015, and they're widely used as a stand-in for public attention. Instagram follower history isn't available.

**Why not Google Trends as the main measure?** Trends scores are 0–100, which is fine for seeing a spike within one player's line. The trouble is everything that comes with that scale:

- **Each query is scaled to its own peak.** A 100 for a backup kicker and a 100 for Patrick Mahomes can mean very different search volumes, so spike sizes can't be compared across players. That comparison is the core of our question (Madden cover vs. All-Pro).
- **The scale depends on the date range.** Request a different window and every number changes. Lining players up means chaining queries through a shared reference term, which adds error.
- **Daily data only comes for short ranges.** Trends switches to weekly values beyond roughly 9 months, and our window (−90 to +180 days, about 270 days) is right at that limit.
- **The values come from a sample.** The same query can return slightly different numbers on different days, and low-search players come back noisy or full of zeros.
- **There's no official API.** The unofficial `pytrends` library gets rate-limited and breaks often.

Wikipedia pageviews avoid all of these: actual daily counts from a stable, official API that are comparable across players. Trends is still useful as an optional **robustness check**. If Madden covers also show a bigger, longer spike there, the result doesn't depend on Wikipedia's particular audience.

In [1]:
import json
import re
import time
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
import lxml.html as LH

DATA = Path("data")
(DATA / "raw").mkdir(parents=True, exist_ok=True)

# Wikimedia asks automated clients to identify themselves with a User-Agent.
HEADERS = {"User-Agent": "madden-curse-research/0.1 (personal data project)"}

PRE_DAYS, POST_DAYS = 90, 180          # window collected around each event
PAGEVIEWS_START = pd.Timestamp("2015-07-01")  # first day the pageviews API has data


def polite_get(url, params=None, max_tries=6):
    # GET with retries. Wikimedia rate-limits anonymous clients (HTTP 429),
    # so on a 429 or 5xx we wait (honouring Retry-After) and try again.
    for attempt in range(max_tries):
        r = requests.get(url, params=params, headers=HEADERS, timeout=30)
        if r.status_code == 429 or r.status_code >= 500:
            wait = int(r.headers.get("Retry-After", 0)) or 2 ** (attempt + 2)
            print(f"  HTTP {r.status_code}; waiting {wait}s before retrying")
            time.sleep(wait)
            continue
        return r
    r.raise_for_status()

## 1. Madden cover announcements

There's no dataset of these dates, so I entered them by hand. Each one was checked against news coverage from the day it happened, and the `source` column links to one of those articles.

Things to watch for (recorded in `notes`):

- **Madden 16 (Odell Beckham Jr.)** was announced in May 2015, before Wikipedia's page-view data begins, so it **can't be used**. The cover was also a fan vote, so Beckham's name was already in the news for about two weeks beforehand.
- **Madden 19:** Antonio Brown was announced for the standard edition. Terrell Owens (retired) was on the Hall of Fame edition, and he's left out.
- **Madden 20:** Mahomes was announced on **NFL Draft night**, which is a very busy news day.
- **Madden 21:** Lamar Jackson told the public himself in April 2020, two months before the official reveal. The official date had also been pushed back from June 1.
- **Madden 22** had two cover athletes (Brady and Mahomes), so it gives us two events.
- **Madden 23** honored John Madden after his death and had no active player, so it's left out.

In [2]:
covers = pd.DataFrame([
    # game, player, wiki_title, announce_date, include, source, notes
    ("Madden NFL 16", "Odell Beckham Jr.", "Odell_Beckham_Jr.", "2015-05-13", False,
     "https://www.cbssports.com/nfl/news/madden-nfl-16-cover-odell-beckham-jr-makes-history-beats-out-gronk",
     "Before pageviews API start (2015-07-01); fan vote publicized finalists ~2 weeks earlier"),
    ("Madden NFL 17", "Rob Gronkowski", "Rob_Gronkowski", "2016-05-12", True,
     "https://www.espn.com/nfl/story/_/id/15520923/new-england-patriots-te-rob-gronkowski-cover-madden-nfl-17",
     "Revealed live on SportsCenter"),
    ("Madden NFL 18", "Tom Brady", "Tom_Brady", "2017-05-12", True,
     "https://bleacherreport.com/articles/2709288-tom-brady-announced-as-madden-18-cover-athlete", ""),
    ("Madden NFL 19", "Antonio Brown", "Antonio_Brown", "2018-07-18", True,
     "https://www.gameinformer.com/2018/07/18/antonio-brown-is-madden-nfl-19s-cover-star",
     "Standard edition; Terrell Owens (retired) on HOF edition, excluded"),
    ("Madden NFL 20", "Patrick Mahomes", "Patrick_Mahomes", "2019-04-25", True,
     "https://shacknews.com/article/111569/chiefs-mvp-qb-patrick-mahomes-lands-on-madden-nfl-20-cover",
     "Announced on NFL Draft night (busy news day)"),
    ("Madden NFL 21", "Lamar Jackson", "Lamar_Jackson", "2020-06-16", True,
     "https://www.nfl.com/news/lamar-jackson-revealed-as-madden-nfl-21-cover-athlete",
     "Jackson self-leaked in April 2020; reveal delayed from June 1"),
    ("Madden NFL 22", "Tom Brady", "Tom_Brady", "2021-06-17", True,
     "https://www.si.com/nfl/2021/06/17/tom-brady-patrick-mahomes-madden-22-cover-stars",
     "Shared cover; teased June 14"),
    ("Madden NFL 22", "Patrick Mahomes", "Patrick_Mahomes", "2021-06-17", True,
     "https://www.si.com/nfl/2021/06/17/tom-brady-patrick-mahomes-madden-22-cover-stars",
     "Shared cover; teased June 14"),
    ("Madden NFL 23", "John Madden", "John_Madden", "2022-06-08", False,
     "", "Memorial cover, not an active player; excluded (date not verified)"),
    ("Madden NFL 24", "Josh Allen", "Josh_Allen", "2023-06-07", True,
     "https://www.cbssports.com/nfl/news/madden-nfl-24-cover-reveal-josh-allen-becomes-first-bills-player-to-be-featured-on-cover-of-game", ""),
    ("Madden NFL 25", "Christian McCaffrey", "Christian_McCaffrey", "2024-06-11", True,
     "https://operationsports.com/madden-nfl-25-cover-athlete-is-christian-mccaffrey-releases-august-16",
     "EA teased reveal date on June 7"),
    ("Madden NFL 26", "Saquon Barkley", "Saquon_Barkley", "2025-06-02", True,
     "https://www.on3.com/pro/news/saquon-barkley-revealed-as-madden-nfl-26-cover-athlete", ""),
    ("Madden NFL 27", "Caleb Williams", "Caleb_Williams", "2026-06-03", True,
     "https://www.on3.com/pro/news/ea-sports-reveals-official-madden-nfl-27-cover-featuring-bears-qb-caleb-williams", ""),
], columns=["game", "player", "wiki_title", "event_date", "include", "source", "notes"])
covers["event_date"] = pd.to_datetime(covers["event_date"])
covers["event_type"] = "madden_cover"
covers

,game,player,wiki_title,event_date,include,source,notes,event_type
0,Madden NFL 16,Odell Beckham Jr.,Odell_Beckham_Jr.,2015-05-13,False,https://www.cbssports.com/nfl/news/madden-nfl-...,Before pageviews API start (2015-07-01); fan v...,madden_cover
1,Madden NFL 17,Rob Gronkowski,Rob_Gronkowski,2016-05-12,True,https://www.espn.com/nfl/story/_/id/15520923/n...,Revealed live on SportsCenter,madden_cover
2,Madden NFL 18,Tom Brady,Tom_Brady,2017-05-12,True,https://bleacherreport.com/articles/2709288-to...,,madden_cover
3,Madden NFL 19,Antonio Brown,Antonio_Brown,2018-07-18,True,https://www.gameinformer.com/2018/07/18/antoni...,Standard edition; Terrell Owens (retired) on H...,madden_cover
4,Madden NFL 20,Patrick Mahomes,Patrick_Mahomes,2019-04-25,True,https://shacknews.com/article/111569/chiefs-mv...,Announced on NFL Draft night (busy news day),madden_cover
5,Madden NFL 21,Lamar Jackson,Lamar_Jackson,2020-06-16,True,https://www.nfl.com/news/lamar-jackson-reveale...,Jackson self-leaked in April 2020; reveal dela...,madden_cover
6,Madden NFL 22,Tom Brady,Tom_Brady,2021-06-17,True,https://www.si.com/nfl/2021/06/17/tom-brady-pa...,Shared cover; teased June 14,madden_cover
7,Madden NFL 22,Patrick Mahomes,Patrick_Mahomes,2021-06-17,True,https://www.si.com/nfl/2021/06/17/tom-brady-pa...,Shared cover; teased June 14,madden_cover
8,Madden NFL 23,John Madden,John_Madden,2022-06-08,False,,"Memorial cover, not an active player; excluded...",madden_cover
9,Madden NFL 24,Josh Allen,Josh_Allen,2023-06-07,True,https://www.cbssports.com/nfl/news/madden-nfl-...,,madden_cover


## 2. AP First-Team All-Pro announcements

**Dates.** The AP announces its team in early January. I checked each date against AP coverage from the day.

**Players.** These come from each season's Wikipedia "All-Pro Team" page. We keep only players marked **AP** on the **first team**. The same pages also list other selectors (PFWA, Sporting News), and we drop those.

In [3]:
AP_DATES = {  # season -> announcement date, each verified against same-day AP coverage
    2015: ("2016-01-08", "https://www.nbcsports.com/nfl/profootballtalk/rumor-mill/news/2015-associated-press-all-pro-team"),
    2016: ("2017-01-06", "https://www.si.com/nfl/2017/01/06/ap-all-pro-team"),
    2017: ("2018-01-05", "https://www.si.com/nfl/2018/01/05/ap-all-pro-team"),
    2018: ("2019-01-04", "https://en.wikipedia.org/wiki/2018_All-Pro_Team (USA Today ref dated 2019-01-04)"),
    2019: ("2020-01-03", "https://wtkr.com/2020/01/03/2019-ap-nfl-all-pro-team-rosters"),
    2020: ("2021-01-08", "https://www.washingtontimes.com/news/2021/jan/8/2020-nfl-all-pro-team-roster/"),
    2021: ("2022-01-14", "https://www.ksat.com/sports/2022/01/14/5-players-are-unanimous-choices-for-aps-nfl-all-pro-team/"),
    2022: ("2023-01-13", "https://www.si.com/nfl/2023/01/13/nfl-2022-all-pro-teams-announced-by-ap"),
    2023: ("2024-01-12", "https://en.wikipedia.org/wiki/2023_All-Pro_Team"),
    2024: ("2025-01-10", "https://www.washingtontimes.com/news/2025/jan/10/lamar-jackson-repeats-justin-jefferson-ja-marr-cha/"),
    2025: ("2026-01-10", "https://bleacherreport.com/articles/25355183-nfl-all-pro-selections-2025-voting-results-and-full-list-players-announced"),
}


def parse_all_pro_page(season):
    # Return AP first-team players from Wikipedia's '<season> All-Pro Team' page.
    cache_file = DATA / "raw" / f"wiki_{season}_All-Pro_Team.html"   # saved copy of the page
    if not cache_file.exists():
        r = polite_get("https://en.wikipedia.org/w/api.php",
                       params={"action": "parse", "page": f"{season} All-Pro Team", "prop": "text",
                               "format": "json", "formatversion": 2})
        cache_file.write_text(r.json()["parse"]["text"])
    doc = LH.fromstring(cache_file.read_text())
    rows = []
    for table in doc.xpath("//table[contains(@class,'wikitable')]"):
        side = table.xpath("string(.//tr[1])").split()[0]   # Offense / Defense / Special teams
        for tr in table.xpath(".//tr"):
            cells = tr.xpath("./th|./td")                   # [position, first team, second team]
            if len(cells) < 2 or not cells[1].xpath(".//a"):
                continue
            position = cells[0].text_content().strip()
            # Each <br>-separated chunk is one player: 'Player, Team (AP, PFWA, ...)'
            raw = LH.tostring(cells[1], encoding=str)
            for chunk in re.split(r"<br\s*/?>", raw):
                frag = LH.fragment_fromstring(chunk, create_parent="div")
                text = frag.text_content()
                m = re.search(r"\(([^)]*)\)\s*$", text.strip())
                if not m or "AP" not in [s.strip() for s in m.group(1).split(",")]:
                    continue
                link = frag.xpath(".//a[starts-with(@href,'/wiki/')]")
                if link:
                    rows.append({
                        "season": season, "side": side, "position": position,
                        "player": link[0].text_content().strip(),
                        "wiki_title": link[0].get("href").removeprefix("/wiki/"),
                    })
    return rows


ap_rows = []
for season in AP_DATES:
    ap_rows += parse_all_pro_page(season)
    time.sleep(1)

all_pro = pd.DataFrame(ap_rows).drop_duplicates(["season", "wiki_title"])  # e.g. Khalil Mack 2015 at 2 positions
all_pro["event_date"] = pd.to_datetime(all_pro["season"].map(lambda s: AP_DATES[s][0]))
all_pro["source"] = all_pro["season"].map(lambda s: AP_DATES[s][1])
all_pro["event_type"] = "ap_all_pro_1st"
all_pro["include"] = True
all_pro["game"] = ""
all_pro["notes"] = ""
print(all_pro.groupby("season").size().rename("AP 1st-team players"))
all_pro.head()

season
2015    26
2016    28
2017    28
2018    28
2019    26
2020    27
2021    28
2022    28
2023    30
2024    30
2025    31
Name: AP 1st-team players, dtype: int64


,season,side,position,player,wiki_title,event_date,source,event_type,include,game,notes
0,2015,Offense,Quarterback,Cam Newton,Cam_Newton,2016-01-08,https://www.nbcsports.com/nfl/profootballtalk/...,ap_all_pro_1st,True,,
1,2015,Offense,Halfback,Adrian Peterson,Adrian_Peterson,2016-01-08,https://www.nbcsports.com/nfl/profootballtalk/...,ap_all_pro_1st,True,,
2,2015,Offense,Halfback,Doug Martin,Doug_Martin_(running_back),2016-01-08,https://www.nbcsports.com/nfl/profootballtalk/...,ap_all_pro_1st,True,,
3,2015,Offense,Fullback,Mike Tolbert,Mike_Tolbert,2016-01-08,https://www.nbcsports.com/nfl/profootballtalk/...,ap_all_pro_1st,True,,
4,2015,Offense,Tight end,Rob Gronkowski,Rob_Gronkowski,2016-01-08,https://www.nbcsports.com/nfl/profootballtalk/...,ap_all_pro_1st,True,,


**Sanity check.** Each season should have roughly 25–30 players. Some positions take two or three players (wide receivers, cornerbacks), and in some years the AP added positions such as "flex" and "edge rusher." If a season looks wrong, open its Wikipedia page and compare.

## 3. Combine events and resolve Wikipedia titles

A Wikipedia link can point to a **redirect** (e.g. `Le'Veon_Bell` vs `Le%27Veon_Bell`). The pageviews API only counts views of the exact title, so a redirect would show almost zero views. Here we ask Wikipedia for each title's canonical page name.

In [4]:
from urllib.parse import unquote

events = pd.concat([covers, all_pro], ignore_index=True)
events["wiki_title"] = events["wiki_title"].map(unquote)


def resolve_titles(titles):
    # Map each title to its final (post-redirect) Wikipedia title.
    mapping = {}
    titles = list(dict.fromkeys(titles))
    for i in range(0, len(titles), 50):  # API accepts 50 titles per request
        batch = titles[i:i + 50]
        r = polite_get(
            "https://en.wikipedia.org/w/api.php",
            params={"action": "query", "titles": "|".join(t.replace("_", " ") for t in batch),
                    "redirects": 1, "prop": "description", "format": "json", "formatversion": 2},
        ).json()["query"]
        norm = {n["from"]: n["to"] for n in r.get("normalized", [])}
        redir = {d["from"]: d["to"] for d in r.get("redirects", [])}
        desc = {p["title"]: p.get("description", "") for p in r["pages"]}
        for t in batch:
            name = t.replace("_", " ")
            name = norm.get(name, name)
            name = redir.get(name, name)
            mapping[t] = (name.replace(" ", "_"), desc.get(name, ""))
        time.sleep(0.5)
    return mapping


resolved = resolve_titles(events["wiki_title"])
events["wiki_title"] = events["wiki_title"].map(lambda t: resolved[t][0])
desc_lookup = {title: desc for title, desc in resolved.values()}
events["wiki_description"] = events["wiki_title"].map(desc_lookup)

# Every page should be an American football player, which guards against picking up the wrong 'Josh Allen'
events[["player", "wiki_title", "wiki_description"]].drop_duplicates("wiki_title").query(
    "~wiki_description.str.contains('football', case=False, na=False)")

,player,wiki_title,wiki_description


The table above lists every page whose Wikipedia description **doesn't** mention football. Ideally it's empty. Any row that shows up needs a manual check.

## 4. Flag overlapping events

The same player can have several events close together. For example, Mahomes was named All-Pro in January 2019 and announced as the Madden cover star in April 2019. His April numbers would then still include some of the All-Pro bump. We **don't drop** these events here. We only flag them so you can decide what to do in the analysis.

`overlap_days` = number of days to the nearest *other* event for the same player (blank if none).

In [5]:
events = events.sort_values(["wiki_title", "event_date"]).reset_index(drop=True)
events["event_id"] = (events["event_type"] + "_" + events["event_date"].dt.strftime("%Y%m%d")
                      + "_" + events["wiki_title"])


def nearest_other(g):
    d = g["event_date"]
    out = []
    for i, x in d.items():
        others = (d.drop(i) - x).abs().dt.days
        out.append(others.min() if len(others) else pd.NA)
    return pd.Series(out, index=g.index)


events["overlap_days"] = events.groupby("wiki_title", group_keys=False).apply(nearest_other)
events["overlaps_window"] = events["overlap_days"].le(POST_DAYS).fillna(False)

events["window_start"] = events["event_date"] - pd.Timedelta(days=PRE_DAYS)
events["window_end"] = events["event_date"] + pd.Timedelta(days=POST_DAYS)
events.loc[events["window_start"] < PAGEVIEWS_START, "include"] = False

print(events.groupby("event_type")[["include", "overlaps_window"]].sum())
events.query("event_type == 'madden_cover'")[["player", "event_date", "include", "overlap_days"]]

                include  overlaps_window
event_type                              
ap_all_pro_1st      310                5
madden_cover         11                5


,player,event_date,include,overlap_days
26,Antonio Brown,2018-07-18,True,194
42,Caleb Williams,2026-06-03,True,<NA>
60,Christian McCaffrey,2024-06-11,True,151
147,John Madden,2022-06-08,False,<NA>
155,Josh Allen,2023-06-07,True,<NA>
183,Lamar Jackson,2020-06-16,True,165
224,Odell Beckham Jr.,2015-05-13,False,<NA>
226,Patrick Mahomes,2019-04-25,True,111
227,Patrick Mahomes,2021-06-17,True,575
249,Rob Gronkowski,2016-05-12,True,125


## 4b. Track page renames

The pageviews API logs each view under the title the page had **on the day of the view**. So when a page is renamed, all of its earlier views stay under the old name. One example is `Josh_Allen_(quarterback)`, which became `Josh_Allen` on 2023-08-26. If we only ask for the current title, those earlier views look like zero, or like the views of whatever page used that name before.

To fix this we rebuild each page's **title history**:

1. List every title that currently redirects to the page. A rename leaves the old title behind as a redirect.
2. Check each redirect in Wikipedia's move log to see whether, and when, it was moved.
3. Start from today's title and work backwards through the moves that led to it.

When we download page views later, each day is counted under whichever title the article had **on that day**. We don't add all the titles together, because before a rename the current title may have belonged to a different page, such as a disambiguation page. This also keeps us consistent with every other player: none of them get views from redirect titles either.

The move-log lookups are cached in `data/raw/move_log_cache.json`.

In [ ]:
WIKI_API = "https://en.wikipedia.org/w/api.php"
MOVE_CACHE = DATA / "raw" / "move_log_cache.json"
move_log = json.loads(MOVE_CACHE.read_text()) if MOVE_CACHE.exists() else {}
FAR_FUTURE = pd.Timestamp("2100-01-01")


def get_redirects(titles):
    # Every article-space title that currently redirects to each page (50 pages per request).
    out = {t: [] for t in titles}
    for i in range(0, len(titles), 50):
        params = {"action": "query", "prop": "redirects", "rdlimit": "max", "rdnamespace": 0,
                  "titles": "|".join(t.replace("_", " ") for t in titles[i:i + 50]),
                  "format": "json", "formatversion": 2}
        while True:  # long redirect lists come back in several pages ("continue")
            r = polite_get(WIKI_API, params=params).json()
            for p in r["query"]["pages"]:
                out[p["title"].replace(" ", "_")] += [x["title"].replace(" ", "_")
                                                      for x in p.get("redirects", [])]
            if "continue" not in r:
                break
            params.update(r["continue"])
        time.sleep(0.5)
    return out


def moves_from(title):
    # Move-log entries recorded on `title`: a list of (moved_to, timestamp).
    if title not in move_log:
        r = polite_get(WIKI_API, params={"action": "query", "list": "logevents", "letype": "move",
                                         "letitle": title.replace("_", " "), "lelimit": "max",
                                         "format": "json", "formatversion": 2}).json()
        move_log[title] = [(e["params"]["target_title"].replace(" ", "_"), e["timestamp"])
                           for e in r["query"]["logevents"]
                           if "target_title" in e.get("params", {})]
        time.sleep(0.1)
    return move_log[title]


def title_history(current, redirects):
    # [(title, first_day, last_day), ...] oldest first: which title the article lived under when.
    moves = sorted((pd.Timestamp(ts).tz_convert(None).normalize(), old, new)
                   for old in redirects for new, ts in moves_from(old))
    history, holder, until = [], current, FAR_FUTURE
    for day, old, new in reversed(moves):  # walk back from today's title
        if new == holder:
            history.append((holder, day, until))
            holder, until = old, day - pd.Timedelta(days=1)
    history.append((holder, PAGEVIEWS_START, until))
    return history[::-1]


titles = list(events.query("include")["wiki_title"].unique())
redirects = get_redirects(titles)
print(f"{sum(map(len, redirects.values())):,} redirects to check across {len(titles)} pages "
      f"({sum(t not in move_log for r in redirects.values() for t in r):,} not cached yet)")

histories = {}
for n, t in enumerate(titles, 1):
    histories[t] = title_history(t, redirects[t])
    if n % 25 == 0:
        MOVE_CACHE.write_text(json.dumps(move_log))
        print(f"{n}/{len(titles)}")
MOVE_CACHE.write_text(json.dumps(move_log))

events["title_history"] = events["wiki_title"].map(
    lambda t: " → ".join(f"{title} (from {start:%Y-%m-%d})" for title, start, _ in histories[t])
    if t in histories and len(histories[t]) > 1 else "")

renamed = events.query("include and title_history != ''")
print(f"\n{renamed['wiki_title'].nunique()} pages were renamed at some point since July 2015:")
renamed.drop_duplicates("wiki_title")[["player", "title_history"]]

## 5. Download daily page views

We use the [Wikimedia Pageviews API](https://doc.wikimedia.org/generated-data-platform/aqs/analytics-api/reference/page-views.html) with:

- `agent=user`, which **excludes known bots and crawlers**
- `all-access`, which covers desktop, mobile web and the app

Usually that's one request per event. When a page was renamed during an event's window, the window is split at the rename date and each piece is downloaded under the title in use at the time (see 4b). Results are cached to disk, so a re-run doesn't download everything again. A window that runs past today, such as Caleb Williams's, simply stops at the latest day available.

In [6]:
CACHE = DATA / "raw" / "pageviews_cache"
CACHE.mkdir(exist_ok=True)
API = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
       "en.wikipedia/all-access/user/{title}/daily/{start}/{end}")


def fetch_pageviews(title, start, end):
    cache_file = CACHE / f"{quote(title, safe='')}_{start:%Y%m%d}_{end:%Y%m%d}.json"
    if cache_file.exists():
        return pd.read_json(cache_file)
    url = API.format(title=quote(title, safe=""), start=f"{start:%Y%m%d}", end=f"{end:%Y%m%d}")
    r = polite_get(url)
    if r.status_code == 404:  # no data, e.g. page didn't exist yet
        df = pd.DataFrame(columns=["date", "views"])
    else:
        r.raise_for_status()
        items = r.json()["items"]
        df = pd.DataFrame({"date": [i["timestamp"][:8] for i in items],
                           "views": [i["views"] for i in items]})
    df.to_json(cache_file)
    time.sleep(0.1)
    return df


def window_pieces(ev, end):
    # Split the event window by title history: [(title, start, end), ...].
    pieces = []
    for title, first, last in histories[ev.wiki_title]:
        s, e = max(ev.window_start, first), min(end, last)
        if s <= e:
            pieces.append((title, s, e))
    return pieces


frames = []
to_fetch = events.query("include")
for n, ev in enumerate(to_fetch.itertuples(), 1):
    end = min(ev.window_end, pd.Timestamp.today().normalize() - pd.Timedelta(days=1))
    for title, s, e in window_pieces(ev, end):
        pv = fetch_pageviews(title, s, e)
        if len(pv):
            frames.append(pv.assign(event_id=ev.event_id, source_title=title))
    if n % 50 == 0:
        print(f"{n}/{len(to_fetch)}")

pageviews = pd.concat(frames, ignore_index=True)
pageviews["date"] = pd.to_datetime(pageviews["date"].astype(str), format="%Y%m%d")
pageviews = pageviews.merge(events[["event_id", "event_date"]], on="event_id")
pageviews["rel_day"] = (pageviews["date"] - pageviews["event_date"]).dt.days  # 0 = announcement day
pageviews = pageviews[["event_id", "date", "rel_day", "views", "source_title"]]
print(f"{pageviews['event_id'].nunique()} events, {len(pageviews):,} daily rows")

50/321
100/321
150/321
200/321
250/321


  HTTP 429; waiting 58s before retrying


  HTTP 429; waiting 58s before retrying


  HTTP 429; waiting 58s before retrying


300/321
  HTTP 429; waiting 57s before retrying


  HTTP 429; waiting 58s before retrying


  HTTP 429; waiting 58s before retrying


315 events, 85,244 daily rows


## 6. Quality checks

In [7]:
coverage = (pageviews.groupby("event_id")
            .agg(days=("rel_day", "size"), min_rel=("rel_day", "min"), max_rel=("rel_day", "max"),
                 median_views=("views", "median")))
expected = PRE_DAYS + POST_DAYS + 1
print("Events with incomplete windows:")
coverage.query("days < @expected")

Events with incomplete windows:


,days,min_rel,max_rel,median_views
event_id,,,,
ap_all_pro_1st_20160108_Joe_Thomas_(offensive_tackle),212,-89,180,148.0
ap_all_pro_1st_20180105_Greg_Zuerlein,234,-90,178,2.0
ap_all_pro_1st_20190104_Michael_Dickson,270,-90,180,6.0
ap_all_pro_1st_20240112_Kyle_Hamilton,268,-90,180,5.0
madden_cover_20260603_Caleb_Williams,250,-90,159,1027.0


In [8]:
missing = set(events.query("include")["event_id"]) - set(pageviews["event_id"])
print(f"Included events with no page-view data at all: {len(missing)}")
sorted(missing)

Included events with no page-view data at all: 6


['ap_all_pro_1st_20190104_Shaquille_Leonard',
 'ap_all_pro_1st_20200103_Deonte_Harty',
 'ap_all_pro_1st_20210108_Shaquille_Leonard',
 'ap_all_pro_1st_20220114_AJ_Cole_(American_football)',
 'ap_all_pro_1st_20220114_Shaquille_Leonard',
 'ap_all_pro_1st_20240112_AJ_Cole_(American_football)']

**Windows that span a rename.** These events draw on more than one title. Look for a smooth series with no sudden jump or drop on the rename date. A jump there would mean the title history went wrong.

In [ ]:
spans = (pageviews.groupby("event_id")["source_title"].nunique().loc[lambda s: s > 1].index)
print(f"Events whose window spans a rename: {len(spans)}")
(pageviews[pageviews["event_id"].isin(spans)]
 .groupby(["event_id", "source_title"])
 .agg(first_day=("date", "min"), last_day=("date", "max"), median_views=("views", "median")))

## 7. Save

In [9]:
events.to_csv(DATA / "events.csv", index=False)
pageviews.to_csv(DATA / "pageviews_daily.csv", index=False)
print("Saved:")
for f in ["events.csv", "pageviews_daily.csv"]:
    print(f"  data/{f}  ({(DATA / f).stat().st_size / 1e6:.1f} MB)")

Saved:
  data/events.csv  (0.1 MB)
  data/pageviews_daily.csv  (5.0 MB)


## Output files

**`data/events.csv`** has one row per event.

| column | meaning |
|---|---|
| `event_id` | unique key, e.g. `madden_cover_20190425_Patrick_Mahomes` |
| `event_type` | `madden_cover` or `ap_all_pro_1st` |
| `player`, `wiki_title` | name and canonical Wikipedia page |
| `event_date` | announcement date, which is day 0 |
| `game`, `season`, `position`, `side` | context (cover game, or All-Pro season and position) |
| `include` | False when the event is excluded (pre-2015 data, John Madden, etc.) |
| `overlap_days` / `overlaps_window` | days to that player's nearest other event, and whether it falls inside the post-event window |
| `title_history` | earlier titles of the page and when each started being used (blank if never renamed) |
| `source`, `notes` | where the date came from, plus caveats |

**`data/pageviews_daily.csv`** has one row per event per day.

| column | meaning |
|---|---|
| `event_id` | joins to `events.csv` |
| `date` | calendar date |
| `rel_day` | days relative to the announcement (negative = before) |
| `views` | English Wikipedia page views that day, with bots excluded |
| `source_title` | the page title these views were logged under (differs from `wiki_title` before a rename) |